# `aidp-azure-adls` live test — `abfss://` via OAuth client-creds
Sets the Hadoop ABFS OAuth provider and reads a CSV from an ADLS Gen2 container.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
sa     = os.environ['ADLS_STORAGE_ACCOUNT']
cid    = os.environ['ADLS_CLIENT_ID']
secret = os.environ['ADLS_CLIENT_SECRET']
tenant = os.environ['ADLS_TENANT']
host   = f'{sa}.dfs.core.windows.net'
spark.conf.set(f'fs.azure.account.auth.type.{host}',           'OAuth')
spark.conf.set(f'fs.azure.account.oauth.provider.type.{host}', 'org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider')
spark.conf.set(f'fs.azure.account.oauth2.client.id.{host}',     cid)
spark.conf.set(f'fs.azure.account.oauth2.client.secret.{host}', secret)
spark.conf.set(f'fs.azure.account.oauth2.client.endpoint.{host}', f'https://login.microsoftonline.com/{tenant}/oauth2/token')


In [ ]:
container = os.environ['ADLS_CONTAINER']
data_file = os.environ['ADLS_DATA_FILE']
df = (spark.read.format('csv').option('header', True)
        .load(f'abfss://{container}@{sa}.dfs.core.windows.net/{data_file}'))
df.show()


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-azure-adls',
    'auth': 'oauth-clientcreds',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
